In [ ]:

import kagglehub
pratt3000_vctk_corpus_path = kagglehub.dataset_download('pratt3000/vctk-corpus')

print('Data source import complete.')

Mounting files to /kaggle/input/vctk-corpus...
Data source import complete.


In [ ]:
import librosa
print(librosa.__version__)


0.11.0


In [ ]:
import os
import sys
import math
import json
import glob
import random
import numpy as np
import librosa
from collections import defaultdict, Counter
import os
import numpy as np
import librosa
from glob import glob
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt


In [ ]:
DATA_PATH = "/kaggle/input/vctk-corpus/VCTK-Corpus/VCTK-Corpus/wav48"

speakers = sorted(os.listdir(DATA_PATH))

speaker_files = glob(os.path.join(DATA_PATH, speakers[0], "*.wav"))
print("Speaker:",speakers[:5])
print("Number of files:", len(speaker_files))


Speaker: ['p225', 'p226', 'p227', 'p228', 'p229']
Number of files: 231


# Feature Extraction (MFCC)

In [ ]:
NUM_SPEAKERS = 5
N_MFCC = 13
SR = 16000

def extract_features(file_path, n_mfcc=N_MFCC, sr=SR):

    signal, sr = librosa.load(file_path, sr=sr)
    signal, _ = librosa.effects.trim(signal)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=n_mfcc)
    return mfcc.T

speakers = sorted(os.listdir(DATA_PATH))
print("Speakers in dataset:", speakers[:NUM_SPEAKERS])

Speakers in dataset: ['p225', 'p226', 'p227', 'p228', 'p229']


In [ ]:
speaker_features = {}

for speaker in speakers[:NUM_SPEAKERS]:
    speaker_path = os.path.join(DATA_PATH, speaker)
    audio_files = glob(os.path.join(speaker_path, "*.wav"))

    feats = [extract_features(f) for f in audio_files]
    speaker_features[speaker] = np.vstack(feats)

    print(f"Speaker {speaker} feature shape: {speaker_features[speaker].shape}")

example_speaker = speakers[0]
print(f"\nExample speaker: {example_speaker}")
print("MFCC feature array shape:", speaker_features[example_speaker].shape)

Speaker p225 feature shape: (31008, 13)
Speaker p226 feature shape: (48904, 13)
Speaker p227 feature shape: (45319, 13)
Speaker p228 feature shape: (47542, 13)
Speaker p229 feature shape: (38123, 13)

Example speaker: p225
MFCC feature array shape: (31008, 13)


# Implement EM Algorithm for GMM

In [ ]:
def initialize_gmm(X, n_components=4):
    n_samples, n_features = X.shape
    weights = np.ones(n_components) / n_components
    means = X[np.random.choice(n_samples, n_components, replace=False)]
    covariances = np.array([np.cov(X, rowvar=False) + 1e-6*np.eye(n_features) for _ in range(n_components)])
    return weights, means, covariances


In [ ]:
def e_step(X, weights, means, covariances):
    n_samples = X.shape[0]
    n_components = weights.shape[0]
    gamma = np.zeros((n_samples, n_components))

    for k in range(n_components):
        gamma[:, k] = weights[k] * multivariate_normal.pdf(X, mean=means[k], cov=covariances[k])

    gamma /= gamma.sum(axis=1, keepdims=True)
    return gamma


In [ ]:
def m_step(X, gamma):
    n_samples, n_features = X.shape
    n_components = gamma.shape[1]

    N_k = gamma.sum(axis=0)
    weights = N_k / n_samples
    means = np.dot(gamma.T, X) / N_k[:, np.newaxis]

    covariances = np.zeros((n_components, n_features, n_features))
    for k in range(n_components):
        diff = X - means[k]
        covariances[k] = np.dot((gamma[:, k] * diff.T), diff) / N_k[k]
        covariances[k] += 1e-6 * np.eye(n_features)

    return weights, means, covariances


In [ ]:
def train_gmm(X, n_components=4, n_iter=20, tol=1e-3):
    weights, means, covariances = initialize_gmm(X, n_components)
    log_likelihood_prev = None

    for i in range(n_iter):
        gamma = e_step(X, weights, means, covariances)

        weights, means, covariances = m_step(X, gamma)

        likelihoods = np.sum([
            weights[k] * multivariate_normal.pdf(X, mean=means[k], cov=covariances[k])
            for k in range(n_components)
        ], axis=0)
        log_likelihood = np.sum(np.log(likelihoods + 1e-10))

        if log_likelihood_prev is not None and abs(log_likelihood - log_likelihood_prev) < tol:
            break
        log_likelihood_prev = log_likelihood

    return weights, means, covariances


In [ ]:
n_components = 4
speaker_gmms = {}

for speaker in speakers[:NUM_SPEAKERS]:
    print("Training GMM for:", speaker)
    X = speaker_features[speaker]
    weights, means, covariances = train_gmm(X, n_components)
    speaker_gmms[speaker] = (weights, means, covariances)


Training GMM for: p225
Training GMM for: p226
Training GMM for: p227
Training GMM for: p228
Training GMM for: p229


In [ ]:
def predict_speaker(test_features, speaker_gmms):
    best_speaker = None
    max_log_likelihood = -np.inf

    for speaker, (weights, means, covariances) in speaker_gmms.items():
        likelihoods = np.sum([
            weights[k] * multivariate_normal.pdf(test_features, mean=means[k], cov=covariances[k])
            for k in range(len(weights))
        ], axis=0)
        log_likelihood = np.sum(np.log(likelihoods + 1e-10))

        if log_likelihood > max_log_likelihood:
            max_log_likelihood = log_likelihood
            best_speaker = speaker

    return best_speaker


In [ ]:

test_file = glob(os.path.join(DATA_PATH, speakers[0], "*.wav"))[0]
test_feat = extract_features(test_file)

predicted_speaker = predict_speaker(test_feat, speaker_gmms)
print("Predicted speaker:", predicted_speaker)
print("Actual speaker:", speakers[0])


Predicted speaker: p225
Actual speaker: p225
